In [1]:
import os
import numpy as np
import rasterio

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

In [2]:
samples = sorted(os.listdir("samples"))
labels = sorted(os.listdir("labels"))

print("SAMPLES:")
for i in range(5):
    print(samples[i])

print("\nLABELS:")
for i in range(5):
    print(labels[i])

SAMPLES:
S2A_MSIL2A_20171025T150721_N0500_R082_T19QGA_20230910T203157.SAFE_img_0.tiff
S2A_MSIL2A_20171025T150721_N0500_R082_T19QGA_20230910T203157.SAFE_img_1.tiff
S2A_MSIL2A_20171025T150721_N0500_R082_T19QGA_20230910T203157.SAFE_img_10.tiff
S2A_MSIL2A_20171025T150721_N0500_R082_T19QGA_20230910T203157.SAFE_img_100.tiff
S2A_MSIL2A_20171025T150721_N0500_R082_T19QGA_20230910T203157.SAFE_img_101.tiff

LABELS:
S2A_MSIL2A_20171025T150721_N0500_R082_T19QGA_20230910T203157.SAFE_ndvi_0.tiff
S2A_MSIL2A_20171025T150721_N0500_R082_T19QGA_20230910T203157.SAFE_ndvi_1.tiff
S2A_MSIL2A_20171025T150721_N0500_R082_T19QGA_20230910T203157.SAFE_ndvi_10.tiff
S2A_MSIL2A_20171025T150721_N0500_R082_T19QGA_20230910T203157.SAFE_ndvi_100.tiff
S2A_MSIL2A_20171025T150721_N0500_R082_T19QGA_20230910T203157.SAFE_ndvi_101.tiff


In [3]:
# We need to normalize our "ndvi values" since initially our values were just the pixel value. 
# Once weve normalized it our values will be -1 to 1. 
# Once we have our actual NDVI values well make out 2 classes 0 for no vegetation aka ndvi values less than 0.3 and 1 for anything above 0.3 aka vegetation. 
def normalize_ndvi(ndvi):
    ndvi = ndvi.astype(np.float32)
    ndvi = (ndvi - ndvi.min()) / (ndvi.max() - ndvi.min() + 1e-8)
    ndvi = ndvi * 2.0 - 1.0
    return ndvi

def ndvi_to_binary_class(ndvi):
    classes = np.zeros_like(ndvi, dtype=np.uint8)

    # 0 = non-vegetation
    # 1 = vegetation
    classes[ndvi >= 0.3] = 1

    return classes

In [4]:
X_list = []
y_list = []

for sample_name in samples:
    sample_path = os.path.join("samples", sample_name)

    label_name = sample_name.replace("img", "ndvi")
    label_path = os.path.join("labels", label_name)

    if not os.path.exists(label_path):
        print(f"Missing label for {sample_name}")
        continue

    with rasterio.open(sample_path) as src: # Loading our sample
        img = src.read().astype(np.float32)

    with rasterio.open(label_path) as src: # Loading our label 
        lbl = src.read(1)

    ndvi_norm = normalize_ndvi(lbl) # Utilizing our already defined function to normalize each label.
    class_lbl = ndvi_to_binary_class(ndvi_norm) # Forming our classes based on there ndvi values.

    X_list.append(img) 
    y_list.append(class_lbl)

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.uint8)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique classes in y:", np.unique(y))

c:\Users\dr274\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\rasterio\__init__.py:379: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


X shape: (614, 3, 256, 256)
y shape: (614, 256, 256)
Unique classes in y: [0 1]


In [5]:
# Dividing our data set into training data and testing data (Make sure its not memorizing our data and actually learning)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (491, 3, 256, 256)
X_test: (123, 3, 256, 256)
y_train: (491, 256, 256)
y_test: (123, 256, 256)


In [6]:
# Turning our dataset into a dataset of tensors. 
# Define a way our model can actually access these tensors. 
class VegetationDataset(Dataset):
    def __init__(self, images, masks):
        self.images = torch.tensor(images, dtype=torch.float32)
        self.masks = torch.tensor(masks, dtype=torch.long)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.masks[idx]

In [7]:
train_dataset = VegetationDataset(X_train, y_train) # Turns our data set into tensors 
test_dataset = VegetationDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True) # Loads these tensors and makes 4 batches for training and testing.
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

In [8]:
# Built a Simple CNN with 8 layers, we used an encoder - decoder cnn since we were dealing with pixels. The encoder allowed us to recognize the patterns of the pixels more easily and then 
# We utilized a decoder to upsample our image (Aka go back to original size after our encoder shrank it). The decoder allowed us to restore the pixel details
class SimpleSegCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(SimpleSegCNN, self).__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),

            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(32, num_classes, kernel_size=1)
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = SimpleSegCNN(num_classes=3).to(device)

Loss_Function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Using device: cuda


In [10]:
# In this Cell what we do is compute the weights by using a predefined function from sckit learn, but we first flatten our labels since we need to give them a 1D list of all the labels
# Then we set out our lost function utilizing those weights and we also make our optimizer to update our models parameters.

model = SimpleSegCNN(num_classes=2).to(device)

y_flat = y_train.reshape(-1)
classes = np.unique(y_flat)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_flat
)

weights = torch.tensor(weights, dtype=torch.float32).to(device)
print("Class weights:", weights.cpu().numpy())

Loss_Function = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(model)

Class weights: [1.0066942 0.9933942]
SimpleSegCNN(
  (encoder): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU()
    (7): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU()
  )
  (decoder): Sequential(
    (0): Upsample(scale_factor=2.0, mode='bilinear')
    (1): Conv2d(64, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (2): ReLU()
    (3): Conv2d(32, 2, kernel_size=(1, 1), stride=(1, 1))
  )
)


In [11]:
# In this cell we train our model over 20 epochs utilizing mini batches. every prediction made the loss was calculated and displayed. 
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)   # (B, 2, H, W)
        loss = Loss_Function(outputs, masks)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

Epoch [1/20], Loss: 0.5513
Epoch [2/20], Loss: 0.4402
Epoch [3/20], Loss: 0.3607
Epoch [4/20], Loss: 0.3677
Epoch [5/20], Loss: 0.3743
Epoch [6/20], Loss: 0.3606
Epoch [7/20], Loss: 0.3313
Epoch [8/20], Loss: 0.2919
Epoch [9/20], Loss: 0.2907
Epoch [10/20], Loss: 0.3018
Epoch [11/20], Loss: 0.3128
Epoch [12/20], Loss: 0.2868
Epoch [13/20], Loss: 0.2761
Epoch [14/20], Loss: 0.2572
Epoch [15/20], Loss: 0.2686
Epoch [16/20], Loss: 0.2631
Epoch [17/20], Loss: 0.2525
Epoch [18/20], Loss: 0.2615
Epoch [19/20], Loss: 0.2580
Epoch [20/20], Loss: 0.2548


In [12]:
model.eval()

all_preds = []
all_true = []

with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        all_preds.append(preds.cpu().numpy().reshape(-1))
        all_true.append(masks.cpu().numpy().reshape(-1))

y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_true)

# Metrics
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred,
    average="weighted",
    zero_division=0
)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

# Check predictions
print("\nUnique predicted classes:", np.unique(y_pred))
print("Unique true classes:", np.unique(y_true))

# Full report
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, zero_division=0))

Accuracy: 0.871150071058816
Precision: 0.8738189798725566
Recall: 0.871150071058816
F1-score: 0.8706911320060828

Unique predicted classes: [0 1]
Unique true classes: [0 1]

Classification Report:

              precision    recall  f1-score   support

           0       0.85      0.92      0.88   4165283
           1       0.90      0.82      0.86   3895645

    accuracy                           0.87   8060928
   macro avg       0.87      0.87      0.87   8060928
weighted avg       0.87      0.87      0.87   8060928



Still not completely finished will work on it tommorow and get it done. 